# Energy-Based Neural Networks

In this notebook, we'll explore **energy-based models (EBMs)** - a powerful framework that unifies many machine learning approaches including Hopfield networks, Boltzmann machines, and modern generative models.

## What We'll Learn

- How **energy functions** capture relationships between variables
- The connection between **energy** and **probability**
- Classic examples: **Hopfield networks** and **Restricted Boltzmann Machines (RBMs)**
- Why training EBMs is challenging (the partition function problem)
- **Contrastive divergence** - a practical training algorithm
- Building and training a complete RBM from scratch

## Why Energy-Based Models?

Unlike traditional neural networks that directly predict outputs, EBMs learn to assign low energy to "good" configurations and high energy to "bad" ones. This perspective:

- Provides a **unified framework** for understanding many models
- Enables **bidirectional** reasoning (not just input → output)
- Naturally handles **structured prediction** and **generative modeling**
- Connects to physics, optimization, and probabilistic reasoning

## 1. Setup

First, let's import the necessary libraries and configure our environment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

from aiml_notebooks import get_device, set_seed

%load_ext autoreload
%autoreload 2

Set random seed for reproducibility and configure device.

In [ ]:
set_seed(42)
device = get_device()
print(f"Using device: {device}")

## 2. Core Concept: Energy Functions

### What is an Energy Function?

An **energy function** $E(\mathbf{x})$ maps any configuration of variables $\mathbf{x}$ to a scalar value (the "energy").

**Key intuition**: 
- **Low energy** = compatible, likely, or "good" configuration
- **High energy** = incompatible, unlikely, or "bad" configuration

The model "prefers" low-energy states, just like physical systems naturally settle into low-energy configurations.

### From Energy to Probability

We can convert energies to probabilities using the **Gibbs (Boltzmann) distribution**:

$$p(\mathbf{x}) = \frac{e^{-E(\mathbf{x})}}{Z}$$

where $Z = \sum_{\mathbf{x}'} e^{-E(\mathbf{x}')}$ is the **partition function** (normalizing constant).

**Intuition**: 
- $e^{-E(\mathbf{x})}$ is large when $E(\mathbf{x})$ is small (low energy → high probability)
- $Z$ ensures probabilities sum to 1
- Computing $Z$ requires summing over **all possible** configurations (exponentially many!)

### Example: Simple 1D Energy Function

Let's create a simple energy function and visualize how it relates to probability.

In [ ]:
# Define a simple energy function: two wells (minima) at x=-2 and x=2
def energy_1d(x):
    return 0.1 * (x**4 - 8*x**2 + 10)

# Sample points
x = np.linspace(-4, 4, 200)
E = energy_1d(x)

# Convert to probability
p_unnormalized = np.exp(-E)
Z = np.trapz(p_unnormalized, x)  # Approximate partition function
p = p_unnormalized / Z

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(x, E, linewidth=2)
ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('Energy E(x)', fontsize=12)
ax1.set_title('Energy Function', fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='k', linewidth=0.5)

ax2.plot(x, p, linewidth=2, color='orangered')
ax2.set_xlabel('x', fontsize=12)
ax2.set_ylabel('Probability p(x)', fontsize=12)
ax2.set_title('Induced Probability Distribution', fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.fill_between(x, p, alpha=0.3, color='orangered')

plt.tight_layout()
plt.show()

**Key observation**: The energy function has two valleys at $x \approx -2$ and $x \approx 2$. These low-energy regions correspond to **peaks in the probability distribution**. The model naturally assigns high probability to low-energy configurations.

## 3. Visualizing 2D Energy Landscapes

Let's extend to 2D to better visualize how energy landscapes work. We'll create an energy function with multiple "basins" (local minima).

In [ ]:
def energy_2d(x, y):
    """Energy function with three wells (local minima)"""
    # Three Gaussian wells centered at different locations
    e1 = -3 * np.exp(-((x - 1)**2 + (y - 1)**2) / 0.5)
    e2 = -2.5 * np.exp(-((x + 1)**2 + (y - 1)**2) / 0.5)
    e3 = -2 * np.exp(-((x)**2 + (y + 1.5)**2) / 0.5)
    return -(e1 + e2 + e3) + 2  # Negative to make wells into valleys

# Create grid
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
Z = energy_2d(X, Y)

# Visualize energy landscape
fig = plt.figure(figsize=(15, 5))

# 3D surface plot
ax1 = fig.add_subplot(131, projection='3d')
surf = ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('Energy')
ax1.set_title('3D Energy Surface')
fig.colorbar(surf, ax=ax1, shrink=0.5)

# Contour plot
ax2 = fig.add_subplot(132)
contour = ax2.contour(X, Y, Z, levels=20, cmap='viridis')
ax2.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.6)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('Energy Contours')
plt.colorbar(contour, ax=ax2)

# Probability distribution
ax3 = fig.add_subplot(133)
P = np.exp(-Z)
P = P / (np.sum(P) * (x[1] - x[0]) * (y[1] - y[0]))  # Normalize
contour_p = ax3.contour(X, Y, P, levels=15, cmap='Reds')
ax3.contourf(X, Y, P, levels=15, cmap='Reds', alpha=0.6)
ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.set_title('Probability Distribution')
plt.colorbar(contour_p, ax=ax3)

plt.tight_layout()
plt.show()

**What we see**:
- The energy surface has **three valleys** (local minima)
- The probability distribution has **three peaks** at these low-energy regions
- The deepest valley (around $(1, 1)$) has the highest probability
- High-energy "hills" have near-zero probability

## 4. Hopfield Networks: Energy-Based Memory

### What is a Hopfield Network?

A **Hopfield network** is a classic energy-based model that stores patterns as low-energy states. It can:
- **Store** multiple binary patterns
- **Retrieve** patterns from noisy/partial inputs
- Act as an **associative memory**

### Energy Function

For binary states $\mathbf{s} \in \{-1, +1\}^n$, the energy is:

$$E(\mathbf{s}) = -\frac{1}{2} \mathbf{s}^T W \mathbf{s} - \mathbf{b}^T \mathbf{s}$$

where $W$ is a symmetric weight matrix and $\mathbf{b}$ is a bias vector.

**Key insight**: Stored patterns correspond to local minima in the energy landscape.

### Implementation: Simple Hopfield Network

Let's implement a Hopfield network that stores small binary patterns.

In [ ]:
class HopfieldNetwork:
    def __init__(self, n_units):
        self.n_units = n_units
        self.W = np.zeros((n_units, n_units))
    
    def train(self, patterns):
        """
        Store patterns using Hebbian learning rule.
        patterns: list of binary vectors in {-1, +1}
        """
        self.W = np.zeros((self.n_units, self.n_units))
        
        for pattern in patterns:
            # Hebbian rule: W += pattern * pattern^T
            self.W += np.outer(pattern, pattern)
        
        # Zero diagonal (no self-connections)
        np.fill_diagonal(self.W, 0)
        
        # Normalize
        self.W /= len(patterns)
    
    def energy(self, state):
        """Compute energy of a state"""
        return -0.5 * state @ self.W @ state
    
    def recall(self, initial_state, max_iter=100):
        """
        Retrieve pattern from initial state using async updates.
        Returns: (final_state, energy_history)
        """
        state = initial_state.copy()
        energy_history = [self.energy(state)]
        
        for _ in range(max_iter):
            # Asynchronous update: flip one random bit
            i = np.random.randint(self.n_units)
            activation = self.W[i] @ state
            state[i] = 1 if activation >= 0 else -1
            
            energy_history.append(self.energy(state))
            
            # Check convergence
            if len(energy_history) > 2 and energy_history[-1] == energy_history[-2]:
                break
        
        return state, energy_history

**How it works**:
- **Training**: Store patterns using Hebbian rule ("neurons that fire together, wire together")
- **Recall**: Start from noisy input, iteratively update units to minimize energy
- **Convergence**: Energy decreases monotonically until reaching a local minimum

### Example: Storing Simple Patterns

Let's store three 5x5 binary patterns (letters) and test pattern completion.

In [ ]:
# Define three simple 5x5 patterns (T, L, X)
pattern_T = np.array([
    [1, 1, 1, 1, 1],
    [-1, -1, 1, -1, -1],
    [-1, -1, 1, -1, -1],
    [-1, -1, 1, -1, -1],
    [-1, -1, 1, -1, -1]
]).flatten()

pattern_L = np.array([
    [1, -1, -1, -1, -1],
    [1, -1, -1, -1, -1],
    [1, -1, -1, -1, -1],
    [1, -1, -1, -1, -1],
    [1, 1, 1, 1, 1]
]).flatten()

pattern_X = np.array([
    [1, -1, -1, -1, 1],
    [-1, 1, -1, 1, -1],
    [-1, -1, 1, -1, -1],
    [-1, 1, -1, 1, -1],
    [1, -1, -1, -1, 1]
]).flatten()

patterns = [pattern_T, pattern_L, pattern_X]

# Visualize patterns
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
titles = ['Pattern T', 'Pattern L', 'Pattern X']

for i, (pattern, title) in enumerate(zip(patterns, titles)):
    axes[i].imshow(pattern.reshape(5, 5), cmap='RdBu', vmin=-1, vmax=1)
    axes[i].set_title(title)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print("Patterns to store (white = +1, red = -1)")

Train the Hopfield network on these patterns.

In [ ]:
# Create and train network
hopfield = HopfieldNetwork(n_units=25)
hopfield.train(patterns)

print(f"Network trained with {len(patterns)} patterns")
print(f"Weight matrix shape: {hopfield.W.shape}")

Test pattern recall with noisy input. We'll corrupt pattern T and see if the network can recover it.

In [ ]:
# Create noisy version of pattern T (flip 30% of bits)
noisy_pattern = pattern_T.copy()
n_flips = int(0.3 * len(noisy_pattern))
flip_indices = np.random.choice(len(noisy_pattern), n_flips, replace=False)
noisy_pattern[flip_indices] *= -1

# Recall
recalled, energy_history = hopfield.recall(noisy_pattern, max_iter=100)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].imshow(pattern_T.reshape(5, 5), cmap='RdBu', vmin=-1, vmax=1)
axes[0].set_title('Original Pattern')
axes[0].axis('off')

axes[1].imshow(noisy_pattern.reshape(5, 5), cmap='RdBu', vmin=-1, vmax=1)
axes[1].set_title('Noisy Input (30% corrupted)')
axes[1].axis('off')

axes[2].imshow(recalled.reshape(5, 5), cmap='RdBu', vmin=-1, vmax=1)
axes[2].set_title('Recalled Pattern')
axes[2].axis('off')

plt.tight_layout()
plt.show()

# Plot energy convergence
plt.figure(figsize=(10, 4))
plt.plot(energy_history, linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Energy')
plt.title('Energy Minimization During Recall')
plt.grid(True, alpha=0.3)
plt.show()

# Check if recall was perfect
accuracy = np.mean(recalled == pattern_T)
print(f"Recall accuracy: {accuracy:.1%}")

**Key observations**:
- The network successfully **recovers** the original pattern from noisy input
- **Energy decreases** monotonically during recall
- The system settles into a **local energy minimum** (stored pattern)
- This demonstrates **content-addressable memory** - partial input retrieves complete pattern

## 5. Restricted Boltzmann Machines (RBMs)

### From Hopfield to RBMs

**Restricted Boltzmann Machines (RBMs)** extend Hopfield networks by introducing:
- Two layers: **visible** units (data) and **hidden** units (features)
- **No connections within a layer** ("restricted" = bipartite graph)
- Stochastic rather than deterministic updates

### Architecture

```
Hidden layer:  h₁   h₂   h₃  ...  hₘ
                \  / \  / \  /
                 \/   \/   \/
                 /\   /\   /\
                /  \ /  \ /  \
Visible layer: v₁   v₂   v₃  ...  vₙ
```

### Energy Function

For binary units $\mathbf{v} \in \{0,1\}^n$ and $\mathbf{h} \in \{0,1\}^m$:

$$E(\mathbf{v}, \mathbf{h}) = -\mathbf{a}^T \mathbf{v} - \mathbf{b}^T \mathbf{h} - \mathbf{v}^T W \mathbf{h}$$

where:
- $\mathbf{a}$ = visible biases
- $\mathbf{b}$ = hidden biases  
- $W$ = weight matrix connecting visible and hidden units

**Lower energy** = more compatible visible-hidden configuration

### Conditional Independence

The "restricted" structure gives us a key property: **conditional independence**.

Given visible units, all hidden units are independent:
$$p(h_j = 1 | \mathbf{v}) = \sigma(b_j + \mathbf{v}^T W_{:,j})$$

Given hidden units, all visible units are independent:
$$p(v_i = 1 | \mathbf{h}) = \sigma(a_i + W_{i,:} \mathbf{h})$$

where $\sigma(x) = 1/(1 + e^{-x})$ is the sigmoid function.

**This makes sampling efficient!** We can update entire layers in parallel.

### Implementation: RBM Class

In [ ]:
class RBM(nn.Module):
    def __init__(self, n_visible, n_hidden):
        super().__init__()
        self.n_visible = n_visible
        self.n_hidden = n_hidden
        
        # Parameters
        self.W = nn.Parameter(torch.randn(n_visible, n_hidden) * 0.01)
        self.a = nn.Parameter(torch.zeros(n_visible))  # visible bias
        self.b = nn.Parameter(torch.zeros(n_hidden))   # hidden bias
    
    def energy(self, v, h):
        """Compute energy E(v, h)"""
        return -torch.sum(self.a * v, dim=1) - torch.sum(self.b * h, dim=1) - torch.sum(v @ self.W * h, dim=1)
    
    def sample_h_given_v(self, v):
        """Sample hidden units given visible units"""
        # Compute probabilities
        p_h = torch.sigmoid(self.b + v @ self.W)
        # Sample
        h = torch.bernoulli(p_h)
        return h, p_h
    
    def sample_v_given_h(self, h):
        """Sample visible units given hidden units"""
        # Compute probabilities
        p_v = torch.sigmoid(self.a + h @ self.W.t())
        # Sample
        v = torch.bernoulli(p_v)
        return v, p_v
    
    def gibbs_step(self, v):
        """One step of Gibbs sampling: v -> h -> v'"""
        h, _ = self.sample_h_given_v(v)
        v_new, _ = self.sample_v_given_h(h)
        return v_new
    
    def forward(self, v, k=1):
        """k steps of contrastive divergence"""
        # Positive phase
        h0, p_h0 = self.sample_h_given_v(v)
        
        # Negative phase: k steps of Gibbs sampling
        v_k = v.clone()
        for _ in range(k):
            h_k, _ = self.sample_h_given_v(v_k)
            v_k, _ = self.sample_v_given_h(h_k)
        
        # Final hidden activations
        h_k, p_h_k = self.sample_h_given_v(v_k)
        
        return v_k, h0, h_k, p_h0, p_h_k

**Key methods**:
- `sample_h_given_v`: Sample hidden from visible (bottom-up)
- `sample_v_given_h`: Sample visible from hidden (top-down)  
- `gibbs_step`: One full Gibbs sampling step
- `forward`: Implements contrastive divergence (training algorithm)

## 6. The Training Challenge

### Why Training EBMs is Hard

We want to maximize the likelihood of data:

$$\mathcal{L} = \sum_{\mathbf{v} \in \text{data}} \log p(\mathbf{v})$$

where $p(\mathbf{v}) = \frac{1}{Z} \sum_{\mathbf{h}} e^{-E(\mathbf{v}, \mathbf{h})}$

The gradient with respect to weights is:

$$\frac{\partial \mathcal{L}}{\partial W} = \underbrace{\mathbb{E}_{\text{data}}[\mathbf{v} \mathbf{h}^T]}_{\text{positive phase}} - \underbrace{\mathbb{E}_{\text{model}}[\mathbf{v} \mathbf{h}^T]}_{\text{negative phase}}$$

**The problem**: Computing $\mathbb{E}_{\text{model}}$ requires sampling from the model's distribution, which needs running Gibbs sampling until convergence (very slow!).

### Contrastive Divergence: A Practical Solution

**Contrastive Divergence (CD-k)** approximates the negative phase by:
1. Start from a data point $\mathbf{v}_0$
2. Run only $k$ steps of Gibbs sampling (typically $k=1$)
3. Use the result as an approximation of the model distribution

**CD-1 algorithm**:
```
1. Take data sample v₀
2. Compute h₀ ~ p(h|v₀)  [positive phase]
3. Sample v₁ ~ p(v|h₀)   [one Gibbs step]
4. Compute h₁ ~ p(h|v₁)  [negative phase]
5. Update: ΔW ∝ (v₀h₀ᵀ - v₁h₁ᵀ)
```

**Why it works**: Starting from data, a few Gibbs steps quickly moves toward the model's typical configurations. This biased estimate is good enough for learning!

### Training Function

In [ ]:
def train_rbm(rbm, dataloader, lr=0.01, k=1, epochs=10, device='cpu'):
    """
    Train RBM using Contrastive Divergence.
    
    Args:
        rbm: RBM model
        dataloader: DataLoader with binary data
        lr: learning rate
        k: number of Gibbs steps
        epochs: number of training epochs
        device: torch device
    """
    rbm = rbm.to(device)
    optimizer = torch.optim.SGD(rbm.parameters(), lr=lr)
    
    history = {'loss': [], 'reconstruction_error': []}
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        epoch_error = 0.0
        n_batches = 0
        
        for batch in dataloader:
            v0 = batch[0].to(device) if isinstance(batch, list) else batch.to(device)
            batch_size = v0.size(0)
            
            # Contrastive divergence
            v_k, h0, h_k, p_h0, p_h_k = rbm(v0, k=k)
            
            # Compute gradients manually (CD update rule)
            optimizer.zero_grad()
            
            # Positive and negative statistics
            positive_grad = torch.matmul(v0.t(), p_h0) / batch_size
            negative_grad = torch.matmul(v_k.t(), p_h_k) / batch_size
            
            # Update weights
            rbm.W.grad = -(positive_grad - negative_grad)
            rbm.a.grad = -torch.mean(v0 - v_k, dim=0)
            rbm.b.grad = -torch.mean(p_h0 - p_h_k, dim=0)
            
            optimizer.step()
            
            # Track metrics
            loss = torch.mean((v0 - v_k) ** 2)
            epoch_loss += loss.item()
            epoch_error += torch.mean(torch.abs(v0 - v_k)).item()
            n_batches += 1
        
        avg_loss = epoch_loss / n_batches
        avg_error = epoch_error / n_batches
        history['loss'].append(avg_loss)
        history['reconstruction_error'].append(avg_error)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f} - Reconstruction Error: {avg_error:.4f}")
    
    return history

## 7. Training an RBM on Real Data

Let's train an RBM on MNIST digits (binarized for simplicity).

In [ ]:
# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x > 0.5).float())  # Binarize
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f"Loaded {len(train_dataset)} training images")

# Visualize some samples
sample_batch = next(iter(train_loader))[0]
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_batch[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('Sample MNIST Digits (Binarized)')
plt.tight_layout()
plt.show()

Create and train the RBM. We'll use 128 hidden units to learn features from 784-dimensional images.

In [ ]:
# Create RBM
n_visible = 28 * 28  # MNIST images are 28x28
n_hidden = 128

rbm = RBM(n_visible=n_visible, n_hidden=n_hidden)
print(f"Created RBM with {n_visible} visible and {n_hidden} hidden units")
print(f"Total parameters: {sum(p.numel() for p in rbm.parameters()):,}")

Train the RBM using CD-1 (contrastive divergence with 1 Gibbs step).

In [ ]:
# Flatten images for RBM
class FlattenTransform:
    def __call__(self, x):
        return x.view(-1)

# Recreate dataset with flattening
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x > 0.5).float()),
    FlattenTransform()
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Train
history = train_rbm(rbm, train_loader, lr=0.1, k=1, epochs=20, device=device)

print("\nTraining complete!")

Visualize the training progress.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history['loss'], linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Reconstruction Loss', fontsize=12)
ax1.set_title('Training Loss', fontsize=13)
ax1.grid(True, alpha=0.3)

ax2.plot(history['reconstruction_error'], linewidth=2, color='orangered')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Mean Absolute Error', fontsize=12)
ax2.set_title('Reconstruction Error', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**What we observe**: Both loss and reconstruction error decrease, indicating the RBM is learning to capture the structure of MNIST digits.

## 8. Visualizing What the RBM Learned

### Learned Features (Weight Visualization)

Each hidden unit learns to detect a particular feature. We can visualize these features by looking at the weights.

In [ ]:
# Visualize weights of first 64 hidden units
weights = rbm.W.detach().cpu().numpy()  # Shape: (n_visible, n_hidden)

fig, axes = plt.subplots(8, 8, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    if i < n_hidden:
        w = weights[:, i].reshape(28, 28)
        ax.imshow(w, cmap='RdBu', vmin=-0.2, vmax=0.2)
    ax.axis('off')

plt.suptitle('Learned Features (Weight Columns)', fontsize=14)
plt.tight_layout()
plt.show()

**What we see**: Each hidden unit learns to detect **local features** like edges, strokes, and curve patterns that appear in digits. These are the building blocks the RBM uses to represent images.

### Reconstruction Test

Let's see how well the RBM can reconstruct input images.

In [ ]:
# Get test samples
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)
test_batch = next(iter(test_loader))[0].to(device)

# Reconstruct
with torch.no_grad():
    h, _ = rbm.sample_h_given_v(test_batch)
    v_recon, _ = rbm.sample_v_given_h(h)

# Visualize
n_display = 8
fig, axes = plt.subplots(2, n_display, figsize=(14, 3.5))

for i in range(n_display):
    # Original
    axes[0, i].imshow(test_batch[i].cpu().reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Original', fontsize=11, loc='left')
    
    # Reconstruction
    axes[1, i].imshow(v_recon[i].cpu().reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Reconstructed', fontsize=11, loc='left')

plt.suptitle('RBM Reconstruction Quality', fontsize=13)
plt.tight_layout()
plt.show()

**Key insight**: The RBM captures the essential structure of digits, though reconstructions are somewhat "blurry" (compressed through the bottleneck of 128 hidden units).

### Sampling from the Model

We can generate new digit-like images by sampling from the learned distribution using Gibbs sampling.

In [ ]:
# Start from random binary image
v_sample = torch.bernoulli(torch.ones(16, n_visible) * 0.5).to(device)

# Run Gibbs sampling for many steps
n_steps = 1000
with torch.no_grad():
    for _ in range(n_steps):
        v_sample = rbm.gibbs_step(v_sample)

# Visualize generated samples
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(v_sample[i].cpu().reshape(28, 28), cmap='gray')
    ax.axis('off')

plt.suptitle(f'Samples Generated by RBM (after {n_steps} Gibbs steps)', fontsize=13)
plt.tight_layout()
plt.show()

**What we see**: After many Gibbs steps, the RBM generates digit-like patterns from random noise. This demonstrates that the model has learned the **distribution** of MNIST digits.

## 9. Experiments and Exploration

### Effect of Number of Hidden Units

Let's explore how the number of hidden units affects reconstruction quality.

In [ ]:
# Train RBMs with different hidden sizes
hidden_sizes = [32, 64, 128, 256]
results = {}

for n_h in hidden_sizes:
    print(f"\nTraining RBM with {n_h} hidden units...")
    rbm_temp = RBM(n_visible=n_visible, n_hidden=n_h)
    hist = train_rbm(rbm_temp, train_loader, lr=0.1, k=1, epochs=10, device=device)
    results[n_h] = hist

print("\nAll experiments complete!")

Compare reconstruction errors across different architectures.

In [ ]:
plt.figure(figsize=(12, 5))

for n_h, hist in results.items():
    plt.plot(hist['reconstruction_error'], label=f'{n_h} hidden units', linewidth=2)

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Reconstruction Error', fontsize=12)
plt.title('Effect of Hidden Layer Size on Reconstruction', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Print final errors
print("\nFinal Reconstruction Errors:")
for n_h in hidden_sizes:
    final_error = results[n_h]['reconstruction_error'][-1]
    print(f"  {n_h:3d} hidden units: {final_error:.4f}")

**Key findings**:
- **More hidden units** = better reconstruction (more capacity)
- **Diminishing returns** beyond ~128 units for MNIST
- Trade-off between model size and reconstruction quality

### Effect of CD-k Steps

Does using more Gibbs steps (CD-2, CD-5) improve learning?

In [ ]:
# Compare different k values
k_values = [1, 2, 5]
cd_results = {}

for k in k_values:
    print(f"\nTraining with CD-{k}...")
    rbm_temp = RBM(n_visible=n_visible, n_hidden=128)
    hist = train_rbm(rbm_temp, train_loader, lr=0.1, k=k, epochs=10, device=device)
    cd_results[k] = hist

print("\nAll CD experiments complete!")

Plot a histogram to understand the distribution.

In [ ]:
plt.figure(figsize=(12, 5))

for k, hist in cd_results.items():
    plt.plot(hist['reconstruction_error'], label=f'CD-{k}', linewidth=2)

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Reconstruction Error', fontsize=12)
plt.title('Effect of Contrastive Divergence Steps', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nFinal Reconstruction Errors:")
for k in k_values:
    final_error = cd_results[k]['reconstruction_error'][-1]
    print(f"  CD-{k}: {final_error:.4f}")

**Insights**:
- **CD-1 often works well** despite being a rough approximation
- More steps (CD-5) can be slightly better but slower
- The practical sweet spot is typically CD-1 or CD-2

## 10. Key Takeaways

### Core Concepts

1. **Energy-Based Models** assign scalar energies to configurations, with low energy = high compatibility/probability

2. **Energy ↔ Probability** via the Gibbs distribution: $p(\mathbf{x}) \propto e^{-E(\mathbf{x})}$

3. **Hopfield Networks** store patterns as energy minima and retrieve them through energy minimization

4. **RBMs** extend this with hidden units that learn features, using a bipartite structure for efficient sampling

5. **Training Challenge**: Computing the partition function $Z$ is intractable for large models

6. **Contrastive Divergence** solves this by approximating the model distribution with a few Gibbs steps from data

### Why Energy-Based Models Matter

- **Unified framework** for understanding many models (Hopfield, RBMs, modern EBMs)
- **Bidirectional reasoning**: can infer missing values, not just predict outputs
- **Generative capability**: can sample new configurations from learned distribution
- **Connection to physics**: energy landscapes provide intuitive understanding

### Modern Developments

While RBMs were historically important (e.g., for pre-training deep networks), modern approaches include:
- **Score-based models** (diffusion models)
- **Neural ODE-based EBMs**
- **Contrastive learning** frameworks
- **Energy-based GANs**

The core principles we've learned remain relevant across all these modern architectures!

### Visualization Summary

Throughout this notebook, we saw how:
- Energy functions create **probability landscapes**
- Hopfield networks **minimize energy** to recall patterns
- RBMs learn **distributed representations** through hidden units
- Contrastive divergence enables **practical training** despite theoretical challenges
- Hidden unit capacity affects **reconstruction quality**

Energy-based thinking provides a powerful lens for understanding learning, inference, and generation in neural networks.